# M6.A3 — 피처 엔지니어링 + 1차 선별

> 산출 근거: `docs/plan/ai/phase_06_model.md` M6.A3 · `docs/research/ai/00_ml_guide_reference.md` PART 2
> 입력: `AI/data_prep/features_build.py` 산출 `processed/features_daily.parquet` (피처 정의 SSOT)
> 선행: `01_eda.ipynb` §9 피처 후보 목록 · 작성: 2026-07-27

**목적** — EDA 근거로 피처 후보를 생성하고(그룹 7종·37열), 단변량 관계·중복·다중공선성(VIF)·
트리 중요도(gain/permutation)로 **1차 선별(keep/hold/drop)** 을 확정해 M6.A5(베이스라인 비교)의 입력을 만든다.

**누수 방지** — 타깃 유래 피처는 전부 `shift(1)` 이후, 유동인구는 전월 값, 결측 보간·이상치 처리는
하지 않는다(M6.A4에서 train 기준 확정). 기상은 학습 시 실측 — 운영 서빙은 예보로 대체(Phase 7).

**실행 방법**
```bash
cd AI/data_prep && python features_build.py   # 피처 테이블 재생성
cd ../notebooks && jupyter nbconvert --to notebook --execute --inplace 02_features.ipynb
```

> 🔒 **공개 저장소 데이터 정책** — 실매장 매출의 절대 금액(원 단위 수치·그림·표)은 커밋하지 않는다. 본 노트북은 **출력 제거 상태로 추적**되며, 본문 서술의 금액은 비율·배수로 대체했다. 전체 수치·그림은 로컬 재실행으로 전량 재현된다 (실행법: `AI/README.md`).

## 판정 요약 (TL;DR)

1. **피처 후보 37열 생성**(dow·month 원핫 전개 시 모델 행렬 42열) — 그룹 7종: 달력 7·학사 7·기상 9·
   타깃 lag/rolling 8·regime 3·유동인구 1·메뉴 집계 2. 정의는 `features_build.py`가 SSOT.
2. **1차 선별: keep 14열(모델 행렬 20열) / hold 7열 / drop 16열** — 선별 셋으로 재학습 시
   5-fold 평균 MAE **-3.5%**: 후보 절반을 덜어내도 성능이 오히려 좋아짐(§7).
3. **설계 발견** — `lag7_sales`는 `lag_dow_sales`와 지지 구간에서 **완전 동일**(7일 전 영업일 = 같은 요일이므로
   정의상 부분집합, 213/213행 일치) → 병합. 기온 min/max/range 완전 공선(VIF ∞), 강수 3종 무기여 재확인(EDA §7과 일치).
4. **프록시 경고 2건** — `floating_prev_m`은 고유값 7개뿐이라 사실상 **월 더미 프록시**(permutation 상위는 착시 가능),
   `roll7_alcohol_share`는 술 비중 0.40→0.25 급변으로 **regime 식별자 역할** → 둘 다 hold(재평가 조건 §7).
5. **방법론 주의** — 단조 증가 추세·regime 피처(`days_since_open` 등)는 forward CV에서 검증 구간 값이 전부
   학습 범위 밖이라 **permutation 중요도가 구조적으로 0** → gain·도메인 근거로 판단해야 함(§6).
6. **참고 성능**(선별 셋, 튜닝 없음) — 재개장 fold의 MAE만 나머지 fold의 약 2배(regime 적응 문제).
   모델 선정·튜닝·regime 대응은 M6.A5.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

_here = Path.cwd()
AI_DIR = next(p for p in [_here.parent, _here, _here / "AI"] if (p / "data_prep").exists())
sys.path.insert(0, str(AI_DIR / "data_prep"))
from features_build import ALL_FEATURES, FEATURE_GROUPS  # 피처 정의 SSOT

PAL = {"blue": "#2a78d6", "blue_dark": "#1c5cab", "orange": "#eb6834", "gray": "#d9d8d4", "ink2": "#52514e"}
_installed = {f.name for f in fm.fontManager.ttflist}
plt.rcParams.update({
    "font.family": [f for f in ("AppleGothic", "Apple SD Gothic Neo", "NanumGothic") if f in _installed] or ["sans-serif"],
    "axes.unicode_minus": False, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 100,
})

feat = pd.read_parquet(AI_DIR / "data/processed/features_daily.parquet").set_index("date")
ob = feat[feat.is_open].copy()          # 모델링 행 = 영업일
y_log = np.log1p(ob.total_amount)        # EDA §2.4 — log 변환 후 거의 대칭
PRE_END = "2025-12-20"                    # 휴업 전 구간 경계 (EDA §2.6)

print(f"캘린더 {len(feat)}일 / 모델링 행(영업일) {len(ob)}일 / 피처 후보 {len(ALL_FEATURES)}열")
pd.DataFrame([(g, len(cols), ", ".join(cols)) for g, cols in FEATURE_GROUPS.items()],
             columns=["그룹", "수", "피처"])

### §1 관찰 — 후보 구성

- EDA §9 후보 목록을 전부 구현: 요일·주말, 월(sin/cos 포함), 공휴일(+전날), 학사 5플래그+주차,
  기온 4종+구간 플래그, 강수 3종, 타깃 lag/rolling 8종, regime 3종, 유동인구 전월, 메뉴 집계 2종.
- 모든 타깃 유래 피처는 영업일 시퀀스 기준 `shift(1)` — 재개장 첫날(02-26)의 `lag1_sales`가
  휴업 직전 영업일(12-20) 매출인 것까지 확인(빌더 검증).

In [ ]:
# §2 결측 정량 — 모델링 행 기준 (보간·처리는 M6.A4에서 train 기준으로)
na = ob[ALL_FEATURES].isna().sum()
na = na[na > 0].sort_values(ascending=False)
na_tbl = pd.DataFrame({"NaN 일수": na, "비율%": (na / len(ob) * 100).round(1)})
CAUSE = {"floating_prev_m": "전월 유동인구 누락 월(7개월분)",
         "lag7_sales": "7일 전이 무매출일(휴업·휴무 반영) — §4에서 병합 처리",
         "roll4dow_mean": "같은 요일 4회 워밍업", "roll14_mean": "14영업일 워밍업",
         "roll7_mean": "7영업일 워밍업", "roll7_alcohol_share": "7영업일 워밍업",
         "roll7_atv": "7영업일 워밍업", "lag_dow_sales": "요일별 첫 영업일(직전 같은 요일 없음)",
         "lag1_sales": "첫 영업일(직전 영업일 없음)", "lag1_tx": "첫 영업일",
         "days_gap_prev_open": "첫 영업일", "precip_mm": "기상 원본 결측 1일(2025-08-04)"}
na_tbl["원인"] = [CAUSE.get(i, "") for i in na_tbl.index]
display(na_tbl)
print(f"핵심 lag 셋(lag1·roll7·lag_dow) 완비 행: {ob[['lag1_sales','roll7_mean','lag_dow_sales']].dropna().shape[0]} / {len(ob)}")

### §2 관찰 — 결측 구조

- lag·rolling 워밍업 NaN은 학습 초기 1~4주에 집중 — **행 제거가 아니라 LightGBM 네이티브 NaN 처리**로
  두는 것이 기본안(트리가 결측 분기 학습). 선형 베이스라인 비교 시에만 워밍업 행 제외.
- `lag7_sales`의 NaN 43일은 "7일 전이 무매출일"이라는 **정보성 결측** — 그러나 §4에서 보듯
  값이 있는 구간은 `lag_dow_sales`와 동일하므로 병합 대상.
- `floating_prev_m` NaN 88일(34%)은 원본 누락 7개월 탓 — **M6.A4 보간 후 재평가**(hold).

In [ ]:
# §3 단변량 — Spearman (전 기간 vs 휴업 전) — dow·month는 비단조 범주라 제외(§6에서 평가)
rank_feats = [c for c in ALL_FEATURES if c not in ("dow", "month")]
pre = ob.loc[:PRE_END]
sp = pd.DataFrame({
    "전 기간": [ob.total_amount.corr(ob[c].astype(float), method="spearman") for c in rank_feats],
    "휴업 전": [pre.total_amount.corr(pre[c].astype(float), method="spearman") for c in rank_feats],
}, index=rank_feats)
sp["절대값 max"] = sp.abs().max(axis=1)
sp = sp.sort_values("절대값 max", ascending=False).round(3)
display(sp.head(16))

top = sp.head(16).iloc[::-1]
fig, ax = plt.subplots(figsize=(8.5, 5.2), constrained_layout=True)
ax.barh(top.index, top["전 기간"], height=0.42, color=PAL["blue"], label="전 기간", align="edge")
ax.barh(top.index, top["휴업 전"], height=-0.42, color=PAL["orange"], label="휴업 전", align="edge")
ax.axvline(0, color=PAL["ink2"], lw=0.8)
ax.set_xlabel("Spearman ρ (vs 일 매출)"); ax.set_title("단변량 순위상관 상위 16 — 구간별 비교")
ax.legend(frameon=False, fontsize=9)
plt.show()

### §3 관찰 — 단변량 관계

- **타깃 lag군이 최상위** — roll7_mean 0.57, lag1_tx 0.54, roll14 0.53, lag1_sales 0.48. EDA 자기상관(0.54/0.47)과 정합.
- 기온 -0.40, is_hot -0.40, is_semester 0.39 — EDA §7 구도 유지(기온↔방학 교란 포함).
- **구간 다이버전스가 프록시 탐지기 역할**:
  - `month_sin` 0.36(전체) vs 0.13(휴업 전) — 3월 재개장 효과가 "월"에 실린 것.
  - `roll7_alcohol_share` **-0.23 vs +0.07로 부호 반전** — 업종 개편 후 술 비중 급감(0.40→0.25)을 타는 regime 식별자.
  - regime 피처(days_since_reopen 0.40)는 휴업 전 구간에선 정의상 NaN/상수 — 전 기간 상관은 구조적.
- 최하위: 강수 3종(|ρ|≤0.06), is_holiday_eve 0.07 — EDA §7 "강수 효과 없음" 재확인.

In [ ]:
# §4 피처 간 중복 — |Spearman| ≥ 0.8 쌍
num = ob[rank_feats].astype(float)
cm = num.corr(method="spearman")
pairs = [(a, b, round(cm.loc[a, b], 3))
         for i, a in enumerate(cm.columns) for b in cm.columns[i + 1:] if abs(cm.loc[a, b]) >= 0.8]
dup_tbl = pd.DataFrame(sorted(pairs, key=lambda x: -abs(x[2])), columns=["피처 A", "피처 B", "ρ"])
dup_tbl["처리"] = ["B 제거 — 지지 구간에서 값 완전 동일(213/213행), A가 B의 부분집합",
                 "둘 다 제거 — EDA §7 강수 무기여", "A만 유지 — B는 A의 누적 표현이라 정보 중복",
                 "min 제거 — avg·range로 재구성", "max 제거 — avg·range로 재구성",
                 "roll14 제거 — roll7 유지", "min·max 제거", "둘 다 유지 — 매출·객수는 의미 다른 신호"]
display(dup_tbl)

j = ob[["lag7_sales", "lag_dow_sales"]].dropna()
print(f"검증 — lag7_sales 값 존재 {len(j)}행 중 lag_dow_sales와 동일: {(j.lag7_sales == j.lag_dow_sales).sum()}행")

### §4 관찰 — 중복 쌍 처리

- `lag7_sales` ↔ `lag_dow_sales` ρ=1.0의 정체: 캘린더 7일 전이 영업일이면 그날은 **같은 요일**이므로
  두 정의가 일치한다. lag7은 "7일 전이 휴무면 NaN"인 부분집합일 뿐 → **lag_dow_sales로 병합**.
- `precip_mm`↔`is_rain`(0.98)은 어차피 둘 다 무기여라 동반 제거. 기온 3형제는 avg+range 2열로 재구성.
- `lag1_sales`↔`lag1_tx`(0.88)는 단가 정보 차이가 있어 **둘 다 유지**하고 §6 중요도로 최종 판단.

In [ ]:
# §5 VIF — 연속형 21열, complete-case (수동 계산: VIF = 1/(1-R²))
from sklearn.linear_model import LinearRegression

cont = ["temp_avg", "temp_min", "temp_max", "temp_range", "precip_mm", "semester_week",
        "month_sin", "month_cos", "lag1_sales", "lag7_sales", "lag_dow_sales", "roll7_mean",
        "roll14_mean", "roll4dow_mean", "lag1_tx", "days_gap_prev_open", "days_since_reopen",
        "days_since_open", "floating_prev_m", "roll7_alcohol_share", "roll7_atv"]
X = ob[cont].dropna()
Xs = (X - X.mean()) / X.std()
vifs = {}
for c in cont:
    r2 = LinearRegression().fit(Xs.drop(columns=c), Xs[c]).score(Xs.drop(columns=c), Xs[c])
    vifs[c] = np.inf if r2 >= 0.9999 else 1 / (1 - r2)
vif_tbl = pd.Series(vifs, name="VIF").sort_values(ascending=False).to_frame().round(1)
print(f"complete-case n={len(X)} (floating 누락·워밍업 제외 탓에 축소 — 해석은 상대 비교용)")
display(vif_tbl[vif_tbl.VIF > 5])

### §5 관찰 — 다중공선성

- **VIF ∞** = 완전 선형 종속: `temp_range = temp_max − temp_min`, `lag7 ≡ lag_dow` — §4 처리안대로 해소.
- **시간 추세 군집**: `days_since_open` 63.7, `days_since_reopen` 44.6, `month_sin/cos` 35 안팎 —
  13개월 데이터에서 "시간이 흐른다"를 4가지로 중복 표현하는 셈. regime 표현은
  `is_post_renewal`+`days_since_reopen` 두 개만 남기고 `days_since_open`·`month`계는 정리(§7).
- lag군 내부(roll7 22.1, roll14 15.7)도 높음 — 트리 모델엔 치명적이지 않으나 roll14 제거로 완화.
- VIF는 선형 관점 지표 — 최종 채택은 §6 트리 중요도와 도메인 근거를 함께 반영.

In [ ]:
# §6 LGBM 중요도 — TimeSeriesSplit 5-fold, gain(fold 정규화 평균) + permutation(검증 구간)
import lightgbm as lgb
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit


def to_matrix(df, feats):
    """dow는 원핫(LightGBM min_data_per_group=100이 소표본 범주 분기를 막음), month는 sin/cos만."""
    X = df[[c for c in feats if c not in ("dow", "month")]].copy()
    X = pd.concat([X, pd.get_dummies(df["dow"], prefix="dow")], axis=1)
    b = X.select_dtypes(bool).columns
    X[b] = X[b].astype(int)
    return X


def cv_run(X, y, do_perm=False):
    gains, perms, maes = [], [], []
    for tr, va in TimeSeriesSplit(n_splits=5).split(X):
        m = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=15,
                              min_child_samples=10, subsample=0.9, colsample_bytree=0.9,
                              random_state=42, verbosity=-1)
        m.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[va], y.iloc[va])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        g = pd.Series(m.booster_.feature_importance("gain"), index=X.columns)
        gains.append(g / g.sum())
        if do_perm:
            pi = permutation_importance(m, X.iloc[va], y.iloc[va], n_repeats=15, random_state=42)
            perms.append(pd.Series(pi.importances_mean, index=X.columns))
        maes.append(np.mean(np.abs(np.expm1(m.predict(X.iloc[va])) - np.expm1(y.iloc[va]))))
    gain = pd.concat(gains, axis=1).mean(axis=1)
    perm = pd.concat(perms, axis=1).mean(axis=1) if do_perm else None
    return gain, perm, maes


X_all = to_matrix(ob, ALL_FEATURES)
gain_a, perm_a, mae_a = cv_run(X_all, y_log, do_perm=True)
print(f"전체 후보 {X_all.shape[1]}열 | fold MAE(원): " + " / ".join(f"{v:,.0f}" for v in mae_a)
      + f" | 평균 {np.mean(mae_a):,.0f}")
print("(마지막 fold = 재개장 이후 검증 구간 — regime 적응 문제로 오차 최대. 대응은 M6.A5)")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), constrained_layout=True)
for ax, s, title in [(axes[0], gain_a.sort_values().tail(15) * 100, "gain (fold 평균, %)"),
                     (axes[1], perm_a.sort_values().tail(15), "permutation (검증 구간)")]:
    ax.barh(s.index, s.values, color=PAL["blue"], height=0.6)
    ax.set_title(f"LGBM 중요도 상위 15 — {title}")
    ax.grid(axis="y", visible=False)
plt.show()

both0 = [c for c in X_all.columns if gain_a[c] < 0.001 and perm_a[c] <= 0]
print("gain·permutation 모두 0 수준:", both0)
print(f"dow 더미 7개 합계: gain {gain_a.filter(like='dow_').sum()*100:.1f}% / perm {perm_a.filter(like='dow_').sum():.4f}")

### §6 관찰 — gain·permutation 갈림길 해석 (이 절이 선별 논리의 핵심)

| 사례 | gain | perm | 해석 |
|---|---|---|---|
| `days_since_open` | 15.0% (2위) | ≤0 | **추세 피처의 구조적 한계** — forward CV에서 검증 값이 전부 학습 범위 밖이라 permutation이 0이 됨. 게다가 VIF 63.7 → drop |
| `floating_prev_m` | 2.1% | 0.005 (7위) | 고유값 **7개뿐** = 사실상 월 더미. permutation이 "월 식별" 효과를 잡는 착시 가능 → M6.A4 보간 후 재평가(hold) |
| `roll7_alcohol_share` | 4.3% | 0.006 | §3 부호 반전과 결합하면 regime 식별자 — 명시적 regime 플래그를 이미 두므로 hold |
| `is_holiday` | 0 | 0 | 공휴일 영업일이 **7일뿐**이라 중요도에 안 잡히는 희소 이벤트 — EDA §5 근거(영업률 50%·매출 저조)로 **도메인 keep** |
| dow 더미 | 합 6.2% | 합 0.011 | 목(dow_3)·금(dow_4)이 상위 — EDA §2.5 목요일 피크와 정합 |

- 상위 안정권: `lag1_sales`(perm 1위), `roll7_mean`(gain 1위), `roll7_atv`, `roll4dow_mean`, `lag1_tx`, `is_weekend`(dow에 흡수).
- regime 플래그(is_post_renewal·days_since_reopen)도 추세형이라 perm 0 — **성능이 아니라 EDA §2.6 사실 근거로 keep**
  (마지막 fold의 도드라진 오차가 "regime 정보가 필요하다"의 실증).

In [ ]:
# §7 1차 선별 확정 — keep / hold / drop + 근거
SELECTION = {
    # keep: 모델 투입 1군 (dow는 원핫 7열로 전개 → 모델 행렬 20열)
    "keep": {
        "dow": "요일 원핫 7열 — EDA §2.5 목 피크·일 저점, dow_3·4 중요도 상위",
        "is_holiday": "희소(영업 7일)하나 EDA §5 영업률 50%·매출 저조 — 도메인 근거",
        "semester_week": "학사 효과 연속 표현 (is_semester는 week>0과 동치라 제거)",
        "is_semester_first2w": "개강 첫 2주 스파이크 (EDA §2.2 3월·9월)",
        "temp_avg": "기온 수준 — 단변량 -0.40 (방학 교란은 다변량에서 분리)",
        "temp_range": "일교차 — min/max 완전공선 해소 후 잔여 정보",
        "lag1_sales": "permutation 1위 (0.034)",
        "lag1_tx": "객수 신호 — 매출 lag와 의미 구분 (§4)",
        "lag_dow_sales": "같은 요일 직전 영업일 (lag7_sales 흡수·병합)",
        "roll7_mean": "gain 1위 (21%) — 최근 수준",
        "roll4dow_mean": "요일별 최근 수준 — 주간 주기 보완",
        "roll7_atv": "최근 객단가 — permutation 4위, 운영 상태 신호",
        "is_post_renewal": "업종 개편 사실 (EDA §2.6) — 성능 아닌 사실 근거",
        "days_since_reopen": "재개장 램프업 — is_post_renewal의 연속 보완",
    },
    # hold: 조건 충족 시 재평가
    "hold": {
        "is_exam": "구간 비교(EDA §5)론 -34%인데 일자가 estimated — 학사일정 검수 후 재평가",
        "floating_prev_m": "월 더미 프록시 의심 + 누락 34% — M6.A4 보간 후, 학기 피처와 중복성 검사 후 결정",
        "roll7_alcohol_share": "regime 프록시 의심 — 명시적 regime 플래그와 중복성 비교 후 결정",
        "month_sin": "13개월 표본에서 월≈고유 구간 — 과적합 위험, M6.A5에서 ablation",
        "month_cos": "위와 동일",
        "is_session": "계절학기 — 겨울分이 휴업과 완전 중첩(EDA §5), 식별 불가 상태",
        "days_gap_prev_open": "연휴·휴업 직후 효과 가설 — 유효 표본 적음, M6.A5 ablation",
    },
    # drop: 제거
    "drop": {
        "lag7_sales": "lag_dow_sales와 완전 동일(§4) — 병합",
        "temp_min": "완전공선 (avg+range로 재구성)", "temp_max": "완전공선",
        "is_cold": "temp_avg에서 파생 가능 — gain·perm 0", "is_hot": "동일",
        "precip_mm": "무기여 (EDA §7 + perm 0)", "is_rain": "무기여", "is_rain_heavy": "무기여",
        "roll14_mean": "roll7과 ρ=0.93 — 중복", "days_since_open": "추세 중복 (VIF 63.7)",
        "is_semester": "semester_week>0과 동치", "is_weekend": "dow 원핫과 완전 중복",
        "is_holiday_eve": "단변량 0.07·perm 0", "is_festival": "표본 2일", "is_makeup_week": "약함(0.16)",
        "month": "원값 — sin/cos로 대체 (그마저 hold)",
    },
}
sel_tbl = pd.DataFrame(
    [(grp, f, why) for grp, d in SELECTION.items() for f, why in d.items()],
    columns=["판정", "피처", "근거"])
counts = sel_tbl["판정"].value_counts()
print(f"keep {counts['keep']}열(모델 행렬 20열) / hold {counts['hold']} / drop {counts['drop']} — 계 {len(sel_tbl)}")
display(sel_tbl)

# 선별 셋 검증 — 같은 CV로 재학습해 전체 후보 대비 성능 확인
X_keep = to_matrix(ob, list(SELECTION["keep"]) + ["month"])  # month는 to_matrix가 자동 제외
gain_k, _, mae_k = cv_run(X_keep, y_log)
cmp = pd.DataFrame({"전체 후보 42열": mae_a, "선별 20열": mae_k},
                   index=[f"fold{i+1}" for i in range(5)])
cmp.loc["평균"] = cmp.mean()
display(cmp.round(0).astype(int).map(lambda v: f"{v:,}"))
print(f"평균 MAE {np.mean(mae_a):,.0f} → {np.mean(mae_k):,.0f} 원 ({(np.mean(mae_k)/np.mean(mae_a)-1)*100:+.1f}%)")

### §7 관찰 — 선별 검증

- 후보 42열 → 20열로 절반 넘게 줄여도 **평균 MAE -3.5%** — 제거분이 노이즈였다는 방증.
- fold별로도 열세 없음(재개장 fold는 양쪽 다 최대 — 피처가 아니라 regime 적응의 문제).
- hold 7열은 각자 **재평가 트리거**가 명시됨: 검수(is_exam), M6.A4 보간(floating), M6.A5 ablation(month·share·gap).

## §8 판정·다음 단계

**M6.A3 종료 판정** — 피처 후보 37열(모델 행렬 42열) 생성 + 상관·중복·VIF·트리 중요도 점검 +
**1차 선별 keep 14/hold 7/drop 16 확정, 선별 셋 성능 검증(-3.5%)**. 산출물 충족.

### M6.A4(결측·이상치 규칙)로 넘기는 것

1. `floating_prev_m` 누락 7개월 보간(선형 후보) → 보간 후 hold 재평가
2. `precip_mm` 결측 1일(2025-08-04) — 선형 보간이면 충분
3. lag 워밍업 NaN 처리 방침 — LGBM 네이티브 유지 vs 선형 모델용 행 제외 이원화
4. 타깃 이상치 — EDA §2.4 후보(log 잔차 k=3.0 또는 winsorize)를 **train 구간만으로** 확정
5. Walk-forward fold 경계 — 휴업 구간(2025-12-21~2026-02-25)이 통째로 검증 fold가 되지 않게 설계

### M6.A5(베이스라인 비교)의 입력

- 피처: **keep 14열(모델 행렬 20열)** + hold는 트리거 충족 시 추가
- 타깃: log1p(일 매출), 평가: MAE 중심(MAPE는 소액일 왜곡 — EDA §2.4 주문 1건 날들), fold 설계는 M6.A4 결과
- 본 노트북의 LGBM 수치는 **튜닝 없는 스크리닝용** — 모델 선정 결론으로 인용 금지

### 검수 연계 (담당자)

- 학사일정 시험주간 확정 → `is_exam` hold 해제 여부
- 휴업 사유 확인 → regime 피처(keep 2종) 해석 확정 + 학습 구간 3안(EDA §9) 결정
- 유동인구 누락 월 원본 → 보간 범위 축소 가능